# ACT × ALOHA：キューブ受け渡しを自分で学習する


In [ ]:
# ■インストール
!pip install -q uv
!uv venv -q --python 3.12 /content/lerobot-env
!uv pip install -q --python /content/lerobot-env/bin/python 'lerobot[training,evaluation,aloha]'
!/content/lerobot-env/bin/lerobot-train --help > /dev/null && echo "インストール完了"

In [ ]:
# ■Google Driveに接続
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ■設定

DATASET = "lerobot/aloha_sim_transfer_cube_human"      # 人のデモ（FTG64 と同じ）
# DATASET = "lerobot/aloha_sim_transfer_cube_scripted" # プログラム生成のデモ

# 学習回数
#STEPS = 20000
STEPS = 50000

SAVE_FREQ = 5000
RUN_NAME = "act_transfer_cube_human"

TRAIN_DIR = f"/content/train/{RUN_NAME}"                       # 学習中の保存先（Colab 内）
DRIVE_DIR = f"/content/drive/MyDrive/lerobot_aloha/{RUN_NAME}"  # バックアップ先（Google Drive）
print("学習の保存先:", TRAIN_DIR)
print("バックアップ先:", DRIVE_DIR)

In [ ]:
import os, glob, shutil, subprocess, time

LEROBOT_BIN = "/content/lerobot-env/bin"
LOG_FILE = "/content/train_log.txt"

# ■保存された途中のモデルを Google Drive にコピーする
def backup(include_newest=False):
    # 途中のモデル（checkpoints/005000 など）を Google Drive にコピーする。
    # 書き込み中かもしれない一番新しいものは、学習が終わるまでコピーしない。
    ckpts = sorted(glob.glob(f"{TRAIN_DIR}/checkpoints/[0-9]*"))
    if not include_newest:
        ckpts = ckpts[:-1]
    for ckpt in ckpts:
        dst = f"{DRIVE_DIR}/checkpoints/{os.path.basename(ckpt)}"
        if not os.path.exists(dst):
            shutil.copytree(ckpt, dst + "_tmp")
            os.rename(dst + "_tmp", dst)
            print("Google Drive にバックアップしました:", dst)

# ■学習を実行し、1分ごとに進み具合を表示しながらバックアップする
def run_training(args):
    env = dict(os.environ, MUJOCO_GL="egl")
    with open(LOG_FILE, "w") as log:
        proc = subprocess.Popen([f"{LEROBOT_BIN}/lerobot-train", *args],
                                stdout=log, stderr=subprocess.STDOUT, env=env)
    start = time.time()
    while proc.poll() is None:
        time.sleep(60)
        lines = [l for l in open(LOG_FILE, errors="ignore") if "step:" in l]
        if lines:
            print(f"[{(time.time() - start) / 60:5.1f} 分] {lines[-1].strip()}")
        backup()
    if proc.returncode != 0:
        print("".join(open(LOG_FILE, errors="ignore").readlines()[-30:]))
        raise RuntimeError("学習がエラーで止まりました。上のログを確認してください。")
    backup(include_newest=True)
    print("学習が完了しました")

print("準備できました")

In [ ]:
# ■学習開始
if os.path.exists(TRAIN_DIR):
    print("この RUN_NAME の学習結果がすでにあります。続きから学習するなら手順6、")
    print("最初からやり直すなら手順3で RUN_NAME を変えてください。")
else:
    run_training([
        "--policy.type=act",
        f"--dataset.repo_id={DATASET}",
        "--env.type=aloha",
        "--env.task=AlohaTransferCube-v0",
        f"--steps={STEPS}",
        "--batch_size=8",
        f"--save_freq={SAVE_FREQ}",
        "--env_eval_freq=0",
        "--log_freq=200",
        "--num_workers=2",
        "--policy.device=cuda",
        "--policy.push_to_hub=false",
        "--wandb.enable=false",
        f"--output_dir={TRAIN_DIR}",
        f"--job_name={RUN_NAME}",
    ])

In [ ]:
# ■続き学習
drive_ckpts = sorted(glob.glob(f"{DRIVE_DIR}/checkpoints/[0-9]*"))
if not drive_ckpts:
    print("バックアップが見つかりません。手順5から学習してください。")
else:
    os.makedirs(f"{TRAIN_DIR}/checkpoints", exist_ok=True)
    for ckpt in drive_ckpts:
        dst = f"{TRAIN_DIR}/checkpoints/{os.path.basename(ckpt)}"
        if not os.path.exists(dst):
            shutil.copytree(ckpt, dst)
    last_link = f"{TRAIN_DIR}/checkpoints/last"
    if os.path.lexists(last_link):
        os.remove(last_link)
    os.symlink(os.path.basename(drive_ckpts[-1]), last_link)
    print("再開する途中のモデル:", drive_ckpts[-1])

    run_training([
        "--resume=true",
        f"--config_path={last_link}/pretrained_model/train_config.json",
        f"--steps={STEPS}",
    ])

In [ ]:
# ■ロボットを動かす

EVAL_ALL = False #False:最後のモデルをN_EPISODES 回実行 True:途中のモデル（5,000、10,000…ステップ）もすべて試す
N_EPISODES = 10

ckpt_root = f"{DRIVE_DIR}/checkpoints" if os.path.exists(f"{DRIVE_DIR}/checkpoints") else f"{TRAIN_DIR}/checkpoints"
ckpts = sorted(glob.glob(f"{ckpt_root}/[0-9]*"))
targets = ckpts if EVAL_ALL else ckpts[-1:]
print("試すモデル:", [os.path.basename(c) for c in targets])

for ckpt in targets:
    step = os.path.basename(ckpt)
    out = f"/content/outputs/eval_{RUN_NAME}/{step}"
    print(f"\n===== {step} ステップのモデル =====")
    !rm -rf {out}
    !MUJOCO_GL=egl {LEROBOT_BIN}/lerobot-eval \
      --policy.path={ckpt}/pretrained_model \
      --env.type=aloha \
      --env.task=AlohaTransferCube-v0 \
      --eval.n_episodes={N_EPISODES} \
      --eval.batch_size=1 \
      --eval.use_async_envs=false \
      --policy.device=cuda \
      --seed=1000 \
      --output_dir={out} 2>&1 | grep -E "pc_success|Error|error" | tail -n 3

In [ ]:
# ■成功率表示

import json
print(f"{'ステップ':>10} | 成功率 | 各回の結果")
for info_path in sorted(glob.glob(f"/content/outputs/eval_{RUN_NAME}/*/eval_info.json")):
    step = info_path.split("/")[-2]
    info = json.load(open(info_path))
    rate = info["overall"]["pc_success"]
    successes = info["per_task"][0]["metrics"]["successes"]
    marks = " ".join("○" if s else "×" for s in successes)
    print(f"{step:>10} | {rate:5.1f}% | {marks}")

In [ ]:
# ■動画表示

from base64 import b64encode
from IPython.display import HTML, display

for path in sorted(glob.glob(f"/content/outputs/eval_{RUN_NAME}/*/videos/**/*.mp4", recursive=True)):
    step = path.split(f"eval_{RUN_NAME}/")[1].split("/")[0]
    data = b64encode(open(path, "rb").read()).decode()
    display(HTML(f"<p>{step} ステップ：{os.path.basename(path)}</p>"
                 f"<video width=480 controls src='data:video/mp4;base64,{data}'></video>"))

In [ ]:
# ■動画ダウンロード

from google.colab import files
shutil.make_archive(f"/content/eval_{RUN_NAME}", "zip", f"/content/outputs/eval_{RUN_NAME}")
files.download(f"/content/eval_{RUN_NAME}.zip")